Code to add to the raw dlc h5 data a new 'bodypart' named weigthed, which is the nose bodypart if its likelihood is greater than 0.3 and a weigted sum according to the likelihoods of all other bodyparts if it less than 0.3.
We then can define this bodypart in the NaviGraph to get less NaN values in the result (now we'll get Nan only for row where all values in DC where NaN)

In [1]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt


In [2]:
# Read the h5 file, update the path to the raw h5 DLC outputs
h5_file = pd.read_hdf('/var/home/almogmeir/Documents/M.Sc/Project/DLC/01_23_2025_13NC_rewardSize-01232025170447-0000/01_23_2025_13NC_rewardSize-01232025170447-0000.h5')

In [3]:
h5_file

scorer    DLC_Resnet50_Maze_GPUJan26shuffle1_snapshot_060              \
bodyparts                                            nose               
coords                                                  x           y   
0                                              551.053162  820.800171   
1                                              550.746338  823.011230   
2                                              551.094788  825.726379   
3                                              550.287659  826.767578   
4                                              551.438599  830.332092   
...                                                   ...         ...   
99570                                          149.979324  633.626343   
99571                                          147.265808  632.973816   
99572                                          146.065842  633.263550   
99573                                          143.665802  634.119751   
99574                                          143.847031  633.520752   

scorer                                                               \
bodyparts                  ear_R                              ear_L   
coords    likelihood           x           y likelihood           x   
0           0.963719  528.301575  814.519897   0.755015  536.369385   
1           0.949312  527.678650  814.920776   0.696565  536.408752   
2           0.909568  528.174683  816.390259   0.672737  537.814026   
3           0.909067  529.141174  816.796875   0.732500  539.972290   
4           0.978875  530.129517  819.083496   0.731442  540.441284   
...              ...         ...         ...        ...         ...   
99570       0.451042  157.659805  635.884827   0.592574  162.293564   
99571       0.627250  157.485184  632.545044   0.577966  156.847946   
99572       0.714764  155.888870  628.826599   0.549921  157.236176   
99573       0.789346  153.174393  628.229980   0.615302  155.361176   
99574       0.746379  155.183243  628.412170   0.614255  155.923889   

scorer                                                               \
bodyparts                           mid_body                          
coords              y likelihood           x           y likelihood   
0          800.520752   0.891290  516.564209  805.334595   0.715115   
1          801.105896   0.864759  514.630432  806.205505   0.749939   
2          803.946045   0.816658  514.854248  807.238831   0.822633   
3          805.755188   0.868001  516.662781  807.182312   0.802709   
4          809.419983   0.778919  517.889160  808.178650   0.816582   
...               ...        ...         ...         ...        ...   
99570      633.817261   0.696401  162.585495  635.035583   0.676522   
99571      639.053955   0.741256  165.344910  636.473450   0.692092   
99572      636.989380   0.691890  165.123077  634.616394   0.631989   
99573      641.143066   0.731955  166.874084  636.087585   0.727986   
99574      639.475159   0.713362  166.238617  635.196289   0.695202   

scorer                                        
bodyparts   tail_base                         
coords              x           y likelihood  
0          495.417084  800.760132   0.350238  
1          494.035248  802.663147   0.395960  
2          493.994598  802.972290   0.433174  
3          494.209198  803.293030   0.381222  
4          494.734619  803.439880   0.378159  
...               ...         ...        ...  
99570      175.247192  642.084045   0.404473  
99571      176.712296  643.177795   0.613429  
99572      176.977646  642.598633   0.599292  
99573      177.192291  642.277527   0.661301  
99574      177.053802  641.767761   0.668558  

[99575 rows x 15 columns]

In [4]:
# Update to be the actual file name according to the first index column value of the h5 table
file_name = 'DLC_Resnet50_Maze_GPUJan26shuffle1_snapshot_060'

# Load the DataFrame from the HDF5 file
dlc_raw_h5 = h5_file[file_name]

# Extract the MultiIndex from the first two rows
multi_index = pd.MultiIndex.from_arrays([dlc_raw_h5.iloc[0], dlc_raw_h5.iloc[1]])

# Convert the data to numeric, errors='coerce' will turn non-numeric values into NaN
dlc_raw_h5 = dlc_raw_h5.apply(pd.to_numeric, errors='coerce')

new_index = pd.MultiIndex.from_tuples([
        ('weighted', 'x'),
        ('weighted', 'y'),
        ('weighted', 'likelihood')
    ])

new_index_df = pd.DataFrame(np.nan, index=range(len(dlc_raw_h5)), columns=new_index)


# Concatenate the original DataFrame with the new columns DataFrame
# dlc_raw_h5 = pd.concat([dlc_raw_h5, new_columns], axis=1)

dlc_raw_h5 = pd.concat([dlc_raw_h5, new_index_df], axis=1)
dlc_raw_h5

nose                              ear_R                         \
                x           y likelihood           x           y likelihood   
0      551.053162  820.800171   0.963719  528.301575  814.519897   0.755015   
1      550.746338  823.011230   0.949312  527.678650  814.920776   0.696565   
2      551.094788  825.726379   0.909568  528.174683  816.390259   0.672737   
3      550.287659  826.767578   0.909067  529.141174  816.796875   0.732500   
4      551.438599  830.332092   0.978875  530.129517  819.083496   0.731442   
...           ...         ...        ...         ...         ...        ...   
99570  149.979324  633.626343   0.451042  157.659805  635.884827   0.592574   
99571  147.265808  632.973816   0.627250  157.485184  632.545044   0.577966   
99572  146.065842  633.263550   0.714764  155.888870  628.826599   0.549921   
99573  143.665802  634.119751   0.789346  153.174393  628.229980   0.615302   
99574  143.847031  633.520752   0.746379  155.183243  628.412170   0.614255   

            ear_L                           mid_body                         \
                x           y likelihood           x           y likelihood   
0      536.369385  800.520752   0.891290  516.564209  805.334595   0.715115   
1      536.408752  801.105896   0.864759  514.630432  806.205505   0.749939   
2      537.814026  803.946045   0.816658  514.854248  807.238831   0.822633   
3      539.972290  805.755188   0.868001  516.662781  807.182312   0.802709   
4      540.441284  809.419983   0.778919  517.889160  808.178650   0.816582   
...           ...         ...        ...         ...         ...        ...   
99570  162.293564  633.817261   0.696401  162.585495  635.035583   0.676522   
99571  156.847946  639.053955   0.741256  165.344910  636.473450   0.692092   
99572  157.236176  636.989380   0.691890  165.123077  634.616394   0.631989   
99573  155.361176  641.143066   0.731955  166.874084  636.087585   0.727986   
99574  155.923889  639.475159   0.713362  166.238617  635.196289   0.695202   

        tail_base                        weighted                 
                x           y likelihood        x   y likelihood  
0      495.417084  800.760132   0.350238      NaN NaN        NaN  
1      494.035248  802.663147   0.395960      NaN NaN        NaN  
2      493.994598  802.972290   0.433174      NaN NaN        NaN  
3      494.209198  803.293030   0.381222      NaN NaN        NaN  
4      494.734619  803.439880   0.378159      NaN NaN        NaN  
...           ...         ...        ...      ...  ..        ...  
99570  175.247192  642.084045   0.404473      NaN NaN        NaN  
99571  176.712296  643.177795   0.613429      NaN NaN        NaN  
99572  176.977646  642.598633   0.599292      NaN NaN        NaN  
99573  177.192291  642.277527   0.661301      NaN NaN        NaN  
99574  177.053802  641.767761   0.668558      NaN NaN        NaN  

[99575 rows x 18 columns]

In [5]:
def calculate_weighted_coordinates(row):
    body_parts = ['ear_R', 'ear_L', 'mid_body', 'tail_base']
    total_weight = 0
    weighted_x = 0
    weighted_y = 0
    weighted_likelihood = 0
    likelihoods = []

    for part in body_parts:
        likelihood = row[part]['likelihood']
        likelihoods.append(likelihood)
        if likelihood > 0:
            weighted_x += row[part]['x'] * likelihood
            weighted_y += row[part]['y'] * likelihood
            total_weight += likelihood

    if total_weight > 0:
        weighted_x /= total_weight
        weighted_y /= total_weight
        weighted_likelihood = sum([likelihood * (likelihood / total_weight) for likelihood in likelihoods])
    likelihoods = []

    return (weighted_x, weighted_y, weighted_likelihood)

# # Apply the function to each row in the dataframe
# dlc_raw_h5['weighted'][['x', 'y', 'likelihood']] = dlc_raw_h5.apply(
#     lambda row: calculate_weighted_coordinates(row) if row['nose']['likelihood'] < 0.3 else pd.Series([row['nose']['x'], row['nose']['y'], row['nose']['likelihood']]),
#     axis=1
# )

for i, row in enumerate(dlc_raw_h5.iterrows()):
    if row[1]['nose']['likelihood'] >= 0.3:
        dlc_raw_h5.loc[i, ('weighted', ['x', 'y', 'likelihood'])] = (row[1]['nose']['x'], row[1]['nose']['y'], row[1]['nose']['likelihood'])
    else:
        dlc_raw_h5.loc[i, ('weighted', ['x', 'y', 'likelihood'])] = calculate_weighted_coordinates(row[1])

# Display the updated dataframe
dlc_raw_h5


nose                              ear_R                         \
                x           y likelihood           x           y likelihood   
0      551.053162  820.800171   0.963719  528.301575  814.519897   0.755015   
1      550.746338  823.011230   0.949312  527.678650  814.920776   0.696565   
2      551.094788  825.726379   0.909568  528.174683  816.390259   0.672737   
3      550.287659  826.767578   0.909067  529.141174  816.796875   0.732500   
4      551.438599  830.332092   0.978875  530.129517  819.083496   0.731442   
...           ...         ...        ...         ...         ...        ...   
99570  149.979324  633.626343   0.451042  157.659805  635.884827   0.592574   
99571  147.265808  632.973816   0.627250  157.485184  632.545044   0.577966   
99572  146.065842  633.263550   0.714764  155.888870  628.826599   0.549921   
99573  143.665802  634.119751   0.789346  153.174393  628.229980   0.615302   
99574  143.847031  633.520752   0.746379  155.183243  628.412170   0.614255   

            ear_L                           mid_body                         \
                x           y likelihood           x           y likelihood   
0      536.369385  800.520752   0.891290  516.564209  805.334595   0.715115   
1      536.408752  801.105896   0.864759  514.630432  806.205505   0.749939   
2      537.814026  803.946045   0.816658  514.854248  807.238831   0.822633   
3      539.972290  805.755188   0.868001  516.662781  807.182312   0.802709   
4      540.441284  809.419983   0.778919  517.889160  808.178650   0.816582   
...           ...         ...        ...         ...         ...        ...   
99570  162.293564  633.817261   0.696401  162.585495  635.035583   0.676522   
99571  156.847946  639.053955   0.741256  165.344910  636.473450   0.692092   
99572  157.236176  636.989380   0.691890  165.123077  634.616394   0.631989   
99573  155.361176  641.143066   0.731955  166.874084  636.087585   0.727986   
99574  155.923889  639.475159   0.713362  166.238617  635.196289   0.695202   

        tail_base                           weighted                         
                x           y likelihood           x           y likelihood  
0      495.417084  800.760132   0.350238  551.053162  820.800171   0.963719  
1      494.035248  802.663147   0.395960  550.746338  823.011230   0.949312  
2      493.994598  802.972290   0.433174  551.094788  825.726379   0.909568  
3      494.209198  803.293030   0.381222  550.287659  826.767578   0.909067  
4      494.734619  803.439880   0.378159  551.438599  830.332092   0.978875  
...           ...         ...        ...         ...         ...        ...  
99570  175.247192  642.084045   0.404473  149.979324  633.626343   0.451042  
99571  176.712296  643.177795   0.613429  147.265808  632.973816   0.627250  
99572  176.977646  642.598633   0.599292  146.065842  633.263550   0.714764  
99573  177.192291  642.277527   0.661301  143.665802  634.119751   0.789346  
99574  177.053802  641.767761   0.668558  143.847031  633.520752   0.746379  

[99575 rows x 18 columns]

In [6]:
dlc_raw_h5[dlc_raw_h5['nose']['likelihood'] < 0.3]

nose                              ear_R                         \
                x           y likelihood           x           y likelihood   
156    521.529602  801.636169   0.231167  516.515137  809.538696   0.525858   
157    521.353210  808.555359   0.192811  515.874207  809.453247   0.540495   
158    520.320190  809.340515   0.191756  514.991577  810.074951   0.519821   
159    518.578735  810.155823   0.178591  512.990234  811.614075   0.412372   
160    513.894226  811.165588   0.186717  508.627289  813.442871   0.453599   
...           ...         ...        ...         ...         ...        ...   
99502  158.001740  622.303650   0.104579  179.863617  633.551819   0.170279   
99503  155.915436  616.162842   0.128818  181.796844  646.632812   0.156574   
99504  157.511124  617.741333   0.124735  173.004440  646.743347   0.237881   
99505  157.462494  619.413147   0.153942  170.987228  631.540344   0.259530   
99506  161.643158  623.503418   0.238379  168.823776  630.002808   0.392216   

            ear_L                           mid_body                         \
                x           y likelihood           x           y likelihood   
156    511.208496  803.908752   0.323566  494.313141  810.298157   0.850952   
157    511.422180  804.483093   0.296545  493.607239  809.832642   0.849704   
158    511.033936  804.645264   0.268886  492.910645  809.981995   0.849628   
159    509.882141  806.446655   0.225752  491.112793  811.018250   0.843862   
160    501.308563  805.841553   0.208412  489.109253  810.030518   0.846943   
...           ...         ...        ...         ...         ...        ...   
99502  157.962189  631.890991   0.237512  169.908722  644.496399   0.763186   
99503  155.368927  627.478455   0.332618  168.200104  642.729797   0.880482   
99504  156.090240  626.369141   0.326098  165.269608  641.649902   0.973106   
99505  155.921509  626.273804   0.322485  161.296844  642.967712   0.871248   
99506  154.926758  628.065674   0.517371  158.126495  646.431824   0.790402   

        tail_base                           weighted                         
                x           y likelihood           x           y likelihood  
156    478.644867  810.807922   0.596491  497.707277  809.356573   0.636146  
157    477.971222  810.089722   0.604881  497.037201  809.118764   0.640572  
158    476.973022  807.372375   0.675494  495.324592  808.620859   0.657211  
159    475.938110  807.203064   0.642580  492.763912  809.494218   0.633554  
160    475.172058  808.816040   0.605207  490.509790  810.002042   0.630404  
...           ...         ...        ...         ...         ...        ...  
99502  180.776047  658.718933   0.758060  173.587111  647.567341   0.644111  
99503  176.879120  659.340515   0.610370  169.795235  645.596868   0.647940  
99504  169.509598  657.236877   0.585584  165.895925  644.173180   0.684408  
99505  164.832169  658.559937   0.662084  162.772801  643.900930   0.647071  
99506  162.641647  661.020813   0.780059  160.570913  644.590903   0.667218  

[8371 rows x 18 columns]

In [7]:
# Restore the top level index (scorer) with the file name
h5_file = pd.concat({file_name: dlc_raw_h5}, axis=1)
# Restore the index column names
h5_file.columns.names = ['scorer', 'bodyparts', 'coords']
h5_file

scorer    DLC_Resnet50_Maze_GPUJan26shuffle1_snapshot_060              \
bodyparts                                            nose               
coords                                                  x           y   
0                                              551.053162  820.800171   
1                                              550.746338  823.011230   
2                                              551.094788  825.726379   
3                                              550.287659  826.767578   
4                                              551.438599  830.332092   
...                                                   ...         ...   
99570                                          149.979324  633.626343   
99571                                          147.265808  632.973816   
99572                                          146.065842  633.263550   
99573                                          143.665802  634.119751   
99574                                          143.847031  633.520752   

scorer                                                               \
bodyparts                  ear_R                              ear_L   
coords    likelihood           x           y likelihood           x   
0           0.963719  528.301575  814.519897   0.755015  536.369385   
1           0.949312  527.678650  814.920776   0.696565  536.408752   
2           0.909568  528.174683  816.390259   0.672737  537.814026   
3           0.909067  529.141174  816.796875   0.732500  539.972290   
4           0.978875  530.129517  819.083496   0.731442  540.441284   
...              ...         ...         ...        ...         ...   
99570       0.451042  157.659805  635.884827   0.592574  162.293564   
99571       0.627250  157.485184  632.545044   0.577966  156.847946   
99572       0.714764  155.888870  628.826599   0.549921  157.236176   
99573       0.789346  153.174393  628.229980   0.615302  155.361176   
99574       0.746379  155.183243  628.412170   0.614255  155.923889   

scorer                                                               \
bodyparts                           mid_body                          
coords              y likelihood           x           y likelihood   
0          800.520752   0.891290  516.564209  805.334595   0.715115   
1          801.105896   0.864759  514.630432  806.205505   0.749939   
2          803.946045   0.816658  514.854248  807.238831   0.822633   
3          805.755188   0.868001  516.662781  807.182312   0.802709   
4          809.419983   0.778919  517.889160  808.178650   0.816582   
...               ...        ...         ...         ...        ...   
99570      633.817261   0.696401  162.585495  635.035583   0.676522   
99571      639.053955   0.741256  165.344910  636.473450   0.692092   
99572      636.989380   0.691890  165.123077  634.616394   0.631989   
99573      641.143066   0.731955  166.874084  636.087585   0.727986   
99574      639.475159   0.713362  166.238617  635.196289   0.695202   

scorer                                                                \
bodyparts   tail_base                           weighted               
coords              x           y likelihood           x           y   
0          495.417084  800.760132   0.350238  551.053162  820.800171   
1          494.035248  802.663147   0.395960  550.746338  823.011230   
2          493.994598  802.972290   0.433174  551.094788  825.726379   
3          494.209198  803.293030   0.381222  550.287659  826.767578   
4          494.734619  803.439880   0.378159  551.438599  830.332092   
...               ...         ...        ...         ...         ...   
99570      175.247192  642.084045   0.404473  149.979324  633.626343   
99571      176.712296  643.177795   0.613429  147.265808  632.973816   
99572      176.977646  642.598633   0.599292  146.065842  633.263550   
99573      177.192291  642.277527   0.661301  143.665802  634.119751   
99574      177.053802  641.767761   0.668558  143.8

In [8]:
# Save the updated DataFrame back to the HDF5 file, update path and name as wish
h5_file.to_hdf('/var/home/almogmeir/Documents/M.Sc/Project/DLC/01_23_2025_13NC_rewardSize-01232025170447-0000/01_23_2025_13NC_rewardSize-01232025170447-0000_updated.h5', key=file_name, mode='w')